# Hash Tables

Hash table is a data structure that implements an associative array abstract data type, a structure that can map keys to values.

## Hash Function Requirements

1. Should always map a large key to the same small key (deterministic)
2. Should generate values from 0 to m-1 where m is the table size
3. Should be fast: O(1) for integers, O(len) for strings
4. Should uniformly distribute keys into table slots

**Common hash functions:**
- **Integers:** `h(key) = key % m` -- m should be a prime (fewer common factors → better distribution)
- **Strings:** Weighted sum `(str[0] × x⁰ + str[1] × x¹ + ...) % m` where x is a constant (e.g., 33)
- **Universal hashing:** Pick a hash function randomly from a family of functions

> **Birthday Paradox:** If there are 23 people in a room, the probability that two share a birthday is 50%. With 70 people it's 99.9%. This illustrates why collisions are inevitable -- even with a good hash function, collisions happen sooner than you'd expect.

## Collision Handling

When two keys hash to the same index, we have a collision. There are two main approaches:

![Chaining vs Open Addressing](images/hash-chaining-vs-open.png)

### 1. Chaining

- Use an array of linked lists (or dynamic arrays)
- Each slot contains a list of key-value pairs
- **Time complexity:** O(l) where l is length of chain
- **Space:** Extra space for pointers/references
- **Cache performance:** Not cache-friendly due to pointer chasing

### 2. Open Addressing

- Use single array only, no additional data structures
- **Basic requirement:** number of slots ≥ number of keys
- **Cache performance:** Cache-friendly

## Open Addressing Techniques

### 1. Linear Probing

- **Formula:** `hash(key, i) = (h(key) + i) % m`
- **Problem:** Primary clustering near occupied slots
- **Delete:** Mark slot as "DELETED" instead of empty

### 2. Quadratic Probing

- **Formula:** `h(key, i) = (h(key) + i²) % m`
- **Problem:** Secondary clustering
- **Requirements:** α < 0.5 and m is prime

### 3. Double Hashing

- **Formula:** `h(key, i) = (h1(key) + i*h2(key)) % m`
- **Second hash:** `h2(key) = PRIME - (key % PRIME)`
- **Requirement:** h2(key) must be relatively prime to m and ≠ 0

## Performance Analysis

**Load factor:** α = n/m (should be ≤ 1)

**Unsuccessful search:**
- **Chaining:** 1 + α comparisons
- **Open addressing:** 1/(1 - α) comparisons

**Example:** If α = 0.9 (90% full)
- Chaining: 1.9 comparisons
- Open addressing: 10 comparisons

### Why 1 + α for Chaining?

With n keys in m slots, average chain length = n/m = α. Unsuccessful search: hash to a slot (1 operation) + traverse entire chain (α comparisons) = **1 + α**.

### Why 1/(1-α) for Open Addressing?

Probability first probe hits an occupied slot = α, second ≈ α², etc. Expected probes = 1 + α + α² + ... = **1/(1-α)** (geometric series, valid for α < 1).

## Chaining vs Open Addressing

1. **Capacity:** Chaining never fills up; OA requires resizing when full
2. **Hash sensitivity:** Chaining less sensitive; OA has clustering issues
3. **Cache performance:** Chaining not cache-friendly; OA is cache-friendly
4. **Space:** Chaining needs extra space for pointers; OA may need larger table for same performance

In [ ]:
import unittest

class HashTests(unittest.TestCase):
    def test_chainhash(self):
        chain_hash = ChainHash()
        self.assertIsNone(chain_hash.get_val(2))

        chain_hash.put_val("name", "frodo")
        self.assertEqual(chain_hash.get_val("name"), "frodo")

        chain_hash.put_val("name", "gandalf")
        self.assertEqual(chain_hash.get_val("name"), "gandalf")

        chain_hash.delete_val("name")
        self.assertIsNone(chain_hash.get_val("name"))

class ChainHash:
    def __init__(self, size=7):
        self.size = size
        self.buckets = [[] for _ in range(size)]

In [ ]:
def get_val(self, key):
    bucket = self.buckets[hash(key) % self.size]
    for rec_key, rec_val in bucket:
        if rec_key == key:
            return rec_val
    return None

ChainHash.get_val = get_val

def test_chainhash_get(self):
    chain_hash = ChainHash()
    self.assertIsNone(chain_hash.get_val(2))

HashTests.test_chainhash_get = test_chainhash_get
unittest.main(argv=['', 'HashTests.test_chainhash_get'], verbosity=2, exit=False)

In [ ]:
def put_val(self, key, val):
    bucket = self.buckets[hash(key) % self.size]
    for i, (rec_key, _) in enumerate(bucket):
        if rec_key == key:
            bucket[i] = (key, val)
            return
    bucket.append((key, val))

ChainHash.put_val = put_val

def test_chainhash_put(self):
    chain_hash = ChainHash()
    chain_hash.put_val("name", "frodo")
    self.assertEqual(chain_hash.get_val("name"), "frodo")
    chain_hash.put_val("name", "gandalf")
    self.assertEqual(chain_hash.get_val("name"), "gandalf")

HashTests.test_chainhash_put = test_chainhash_put
unittest.main(argv=['', 'HashTests.test_chainhash_put'], verbosity=2, exit=False)

In [ ]:
def delete_val(self, key):
    bucket = self.buckets[hash(key) % self.size]
    for i, (rec_key, _) in enumerate(bucket):
        if rec_key == key:
            bucket.pop(i)
            break

ChainHash.delete_val = delete_val

def test_chainhash_delete(self):
    chain_hash = ChainHash()
    chain_hash.put_val("name", "frodo")
    chain_hash.delete_val("name")
    self.assertIsNone(chain_hash.get_val("name"))

HashTests.test_chainhash_delete = test_chainhash_delete
unittest.main(argv=['', 'HashTests.test_chainhash_delete'], verbosity=2, exit=False)

In [ ]:
class OpenAddressHash:
    def __init__(self, cap):
        self.cap = cap
        self.buckets = [-1] * cap  # -1 = empty, -2 = deleted
        self.size = 0

    def hash(self, x):
        return x % self.cap

    def insert(self, x):
        if self.size == self.cap:
            return False
        if self.search(x):
            return False
        i = self.hash(x)
        t = self.buckets
        while t[i] not in (-1, -2):
            i = (i + 1) % self.cap
        t[i] = x
        self.size += 1
        return True

In [ ]:
def search(self, x):
    h = self.hash(x)
    t = self.buckets
    i = h
    while t[i] != -1:
        if t[i] == x:
            return True
        i = (i + 1) % self.cap
        if i == h:
            return False
    return False

OpenAddressHash.search = search

def test_open_address_search(self):
    oa_hash = OpenAddressHash(7)
    self.assertFalse(oa_hash.search(10))
    oa_hash.insert(10)
    self.assertTrue(oa_hash.search(10))

HashTests.test_open_address_search = test_open_address_search
unittest.main(argv=['', 'HashTests.test_open_address_search'], verbosity=2, exit=False)

In [ ]:
def remove(self, x):
    h = self.hash(x)
    t = self.buckets
    i = h
    while t[i] != -1:
        if t[i] == x:
            t[i] = -2  # mark as deleted
            return True
        i = (i + 1) % self.cap
        if i == h:
            return False
    return False

OpenAddressHash.remove = remove

def test_open_address_remove(self):
    oa_hash = OpenAddressHash(7)
    oa_hash.insert(10)
    self.assertTrue(oa_hash.remove(10))
    self.assertFalse(oa_hash.search(10))

HashTests.test_open_address_remove = test_open_address_remove
unittest.main(argv=['', 'HashTests.test_open_address_remove'], verbosity=2, exit=False)

# Python Built-in Hash Structures

Python's `dict` uses open addressing (since CPython 3.6). Average O(1) for get/set/delete.

| Built-in | Use case |
|----------|----------|
| `dict` | General key-value mapping |
| `set` | Membership testing, deduplication |
| `defaultdict` | Dict with auto-initialized default values |
| `Counter` | Frequency counting |

In [ ]:
from collections import defaultdict, Counter

# dict -- O(1) average for get, set, delete, 'in'
d = {'a': 1, 'b': 2}
d['c'] = 3
print('a' in d)          # True -- O(1) membership test
print(d.get('z', 0))     # 0 -- safe access with default

# set -- O(1) average for add, remove, 'in'
s = {1, 2, 3}
s.add(4)
print(s & {2, 3, 5})     # {2, 3} -- intersection
print(s | {5, 6})        # {1, 2, 3, 4, 5, 6} -- union

# defaultdict -- auto-creates missing keys with a factory
graph = defaultdict(list)
graph['a'].append('b')   # no KeyError, creates [] first
graph['a'].append('c')
print(dict(graph))       # {'a': ['b', 'c']}

# Counter -- frequency counting in one line
freq = Counter('abracadabra')
print(freq)              # Counter({'a': 5, 'b': 2, 'r': 2, 'c': 1, 'd': 1})
print(freq.most_common(2))  # [('a', 5), ('b', 2)]

# Sets

A set is a hash table that stores only keys (no values). Same O(1) average for add, remove, and membership test.

## Hash Set (`set`)

| Operation | Time | Notes |
|-----------|------|-------|
| `add(x)` | O(1) avg | |
| `remove(x)` | O(1) avg | raises `KeyError` if missing |
| `discard(x)` | O(1) avg | no error if missing |
| `x in s` | O(1) avg | |
| `s \| t` (union) | O(len(s) + len(t)) | |
| `s & t` (intersection) | O(min(len(s), len(t))) | |
| `s - t` (difference) | O(len(s)) | |
| `s ^ t` (symmetric diff) | O(len(s) + len(t)) | elements in either but not both |

In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

print(a | b)   # {1, 2, 3, 4, 5, 6} -- union
print(a & b)   # {3, 4}             -- intersection
print(a - b)   # {1, 2}             -- difference
print(a ^ b)   # {1, 2, 5, 6}       -- symmetric difference
print(a <= b)  # False              -- subset check

# frozenset -- immutable, can be used as dict key or set element
fs = frozenset([1, 2, 3])
d = {fs: 'value'}  # works because frozenset is hashable

## Sorted Set

Python has no built-in sorted set. Java has `TreeSet` (Red-Black tree) and C++ has `std::set` (also RB tree).

| Operation | Hash Set | Sorted Set (BST) |
|-----------|----------|-------------------|
| Add/Remove/Search | O(1) avg | O(log n) |
| Min/Max | O(n) | O(log n) |
| Ordered iteration | O(n log n) sort | O(n) inorder |
| Range query (a..b) | O(n) | O(log n + k) |
| Floor/Ceiling | O(n) | O(log n) |

**When to use sorted set:** When you need ordered operations (min, max, floor, ceiling, range queries) alongside fast insert/delete.

**Python options:**
- `sortedcontainers.SortedSet` -- third-party, B-tree based, excellent performance
- BST (see [Binary Search Tree notebook](../trees/binary-search-tree.ipynb)) -- the underlying data structure
- `bisect` + `list` -- works for small sets, but insert/delete is O(n) due to shifting